# VoyageAI


In [18]:
%pip install -qU voyageai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
EMBEDDING_MODEL = "voyage-large-2-instruct"
DIMENSIONS = 1024

In [20]:
import voyageai
from dotenv import load_dotenv

load_dotenv()

vo = voyageai.Client()

result = vo.embed(["hello world"], model=EMBEDDING_MODEL, input_type="document")

result.embeddings[0][:4]

[0.03380299732089043,
 0.010667615570127964,
 -0.016215115785598755,
 -0.004741863813251257]

In [21]:
assert len(result.embeddings[0]) == DIMENSIONS

In [22]:
from tqdm import tqdm

from typing import Literal


def get_embeddings(texts: list[str], input_type: str = "document") -> list[list[float]]:
    result = vo.embed(texts, model=EMBEDDING_MODEL, input_type=input_type)
    return result.embeddings


def get_embeddings_batched(
    texts: list[str],
    batch_size: int = 128,
    input_type: Literal["document", "query"] = "document",
    show_progress: bool = True,
) -> list[list[float]]:
    all_embeddings = []

    # Create iterable for tqdm
    batches = range(0, len(texts), batch_size)

    # Wrap with tqdm if show_progress is True
    if show_progress:
        batches = tqdm(batches, total=len(batches), desc="Getting embeddings")

    for i in batches:
        batch = texts[i : i + batch_size]
        batch_embeddings = get_embeddings(batch, input_type=input_type)
        all_embeddings.extend(batch_embeddings)

    return all_embeddings

# MongoDB


In [23]:
from dotenv import load_dotenv
import os
import pymongo


load_dotenv()

mongo_client = pymongo.MongoClient(os.getenv("MONGODB_URI"))

db = mongo_client["blogdb"]

collection = db["ai_news"]

collection.find_one()

{'_id': ObjectId('667d1ef6fc45eb48396a6a0c'),
 'date': datetime.datetime(2024, 6, 20, 14, 0),
 'title': "Anthropic's rivalry with OpenAI heats up with its claim new Claude AI surpasses GPT-4o",
 'body': 'Just a month after OpenAI rolled out its latest AI model, GPT-4o, today its rival Anthropic—famously founded by breakaway OpenAI researchers in 2021—said it had developed a new model to top it.',
 'url': 'https://www.msn.com/en-us/news/technology/anthropic-s-rivalry-with-openai-heats-up-with-its-claim-new-claude-ai-surpasses-gpt-4o/ar-BB1oApe1',
 'image': 'https://img-s-msn-com.akamaized.net/tenant/amp/entityid/BB1oAbpV.img?w=2048&h=1366&m=4&q=88',
 'source': 'Fortune on MSN.com',
 'found_at': datetime.datetime(2024, 6, 26, 9, 51, 56, 553000),
 'region': 'wt-wt',
 'domain': 'msn.com',
 'pinecone_indexed': True}

# Pinecone


In [24]:
%pip install -q pinecone-client[grpc]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone()

In [26]:
INDEX_NAME = "ai-news-index"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", region="us-east-1"
        ),  # eu-west-1 not available in free plan
    )

index = pc.Index(INDEX_NAME)

# Upsert all data from MongoDB


In [27]:
from pinecone.core.grpc.protos.vector_service_pb2 import UpsertResponse
from pinecone.grpc import GRPCIndex


def upsert_articles_batched(
    index: GRPCIndex,
    ids: list[str],
    embeddings: list[list[float]],
    metadatas: list[dict],
    batch_size: int = 128,
    show_progress: bool = True,
) -> UpsertResponse:
    assert all(len(vector) == DIMENSIONS for vector in embeddings)
    assert len(ids) == len(embeddings) == len(metadatas)

    # create list of (id, embedding, metadata) tuples to be upserted
    data = list(zip(ids, embeddings, metadatas))
    return index.upsert(
        data,
        batch_size=batch_size,
        show_progress=show_progress,
        async_req=False,
    )  # type: ignore

In [28]:
from pymongo import UpdateOne
from pinecone.core.grpc.protos.vector_service_pb2 import UpsertResponse


def process_and_upsert_articles() -> UpsertResponse | None:
    # Fetch articles that haven't been indexed in Pinecone
    query = {
        "$or": [{"pinecone_indexed": {"$exists": False}}, {"pinecone_indexed": False}]
    }
    articles = list(collection.find(query))

    if not articles:
        print("No new articles to index.")
        return None

    ids = [str(article["_id"]) for article in articles]

    texts = [f"{article['title']}\n\n{article['body']}" for article in articles]
    embeddings = get_embeddings_batched(texts=texts, input_type="document")

    metadatas = [
        {
            "title": article["title"],
            "url": article["url"],
            "body": article["body"],
            "found_at": article["found_at"].timestamp(),
            "date": article["date"].timestamp(),
            "region": article["region"],
        }
        for article in articles
    ]

    # Upsert to Pinecone
    upsert_response = upsert_articles_batched(
        index,
        ids,
        embeddings,
        metadatas,
        show_progress=True,
    )

    # Update MongoDB to mark articles as indexed
    bulk_operations = [
        UpdateOne({"_id": article["_id"]}, {"$set": {"pinecone_indexed": True}})
        for article in articles
    ]
    collection.bulk_write(bulk_operations)

    print(f"Indexed {len(articles)} new articles in Pinecone.")
    return upsert_response


# Run the process
process_and_upsert_articles()


Getting embeddings: 100%|██████████| 50/50 [01:36<00:00,  1.92s/it]


Upserted vectors:   0%|          | 0/6314 [00:00<?, ?it/s]

Indexed 6314 new articles in Pinecone.


upserted_count: 6314

In [29]:
print(index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 42255}},
 'total_vector_count': 42255}


# Test queries


In [17]:
from datetime import datetime, timedelta

QUERY = "tesla FSD V12"

query_embedding = get_embeddings([QUERY], input_type="query")[0]

published_date_start = int((datetime.now() - timedelta(days=10)).timestamp())

query_response = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "date": {"$gte": published_date_start},
    },
)

In [18]:
query_response

{'matches': [{'id': '66825e4fbd1ff5839d2dcc20',
              'metadata': {'body': 'It covers many disruptive technology and '
                                   'trends including Space, Robotics, '
                                   'Artificial Intelligence, Medicine ... A '
                                   'frequent speaker at corporations, he has '
                                   'been a TEDx speaker, a Singularity '
                                   'University speaker and guest at numerous '
                                   'interviews for radio ...',
                           'date': 1719683100.0,
                           'found_at': 1719812367.763,
                           'title': 'Supreme Court Deregulates Tesla FSD and '
                                    'Cars',
                           'url': 'https://www.nextbigfuture.com/2024/06/supreme-court-deregulates-tesla-fsd-and-cars.html'},
              'score': 0.72503275,
              'sparse_values': {'indices'

# Langchain QA


In [19]:
%pip install -qU langchain-pinecone langchain-openai langchain-anthropic langchain-voyageai langchain-text-splitters langchain

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blog-db 0.0.2 requires update<0.0.2,>=0.0.1, which is not installed.

[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_voyageai import VoyageAIEmbeddings

embeddings = VoyageAIEmbeddings(  # type:ignore
    model=EMBEDDING_MODEL,
)

batch size None


In [21]:
from langchain_pinecone import PineconeVectorStore


docsearch = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    text_key="title",
)

docsearch.similarity_search(QUERY, k=3)

[Document(page_content='Supreme Court Deregulates Tesla FSD and Cars', metadata={'body': 'It covers many disruptive technology and trends including Space, Robotics, Artificial Intelligence, Medicine ... A frequent speaker at corporations, he has been a TEDx speaker, a Singularity University speaker and guest at numerous interviews for radio ...', 'date': 1719683100.0, 'found_at': 1719812367.763, 'url': 'https://www.nextbigfuture.com/2024/06/supreme-court-deregulates-tesla-fsd-and-cars.html'}),
 Document(page_content="Elon Musk Suggests It's the End of the Road for the Hardware 3 Autopilot Computer", metadata={'body': 'Elon Musk said that further FSD development would require an upgraded Autopilot computer, meaning that Hardware 3 might have reached its limits', 'date': 1719834240.0, 'found_at': 1719898707.024, 'url': 'https://www.autoevolution.com/news/elon-musk-suggests-it-s-the-end-of-the-road-for-the-hardware-3-autopilot-computer-236318.html'}),
 Document(page_content='Tesla makes p

In [76]:
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from datetime import date

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from operator import itemgetter

gpt_3_5 = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # type:ignore
    temperature=0.0,
)
haiku = ChatAnthropic(model_name="claude-3-haiku-20240307", temperature=0.0)  # type: ignore
sonnet3_5 = ChatAnthropic(model_name="claude-3-5-sonnet-20240620", temperature=0.0)  # type: ignore

llm = sonnet3_5


prompt = PromptTemplate.from_template(
    "You are an assistant for question-answering about the latest AI news. "
    "Use the following pieces of retrieved news articles to answer the question. "
    "You are talking to an experienced auidence in AI. "
    "If you don't know the answer, just say that you don't know. "
    "Format your answer in markdown and add inline hyperlinks."
    "Do not start your answer with 'Based on the provided context', or similar phrases. "
    f"Today is {date.today()} and below are the latest news on AI. \n"
    "Question : {question}\n"
    "Context : \n{context}\n\n"
    "Answer:",
)


def format_docs(docs: list[Document], separator: str = "\n\n") -> str:
    prompt = PromptTemplate.from_template(
        "{page_content} - (Published on {date})\n{url}\n{body}"
    )

    return separator.join(format_document(doc, prompt) for doc in docs)


def deduplicate_docs(docs: list[Document]) -> list[Document]:
    """Deduplicate documents based on their page_content"""

    # Create a dictionary to store unique documents
    unique_docs = {}

    for doc in docs:
        # Use the page_content as the key
        if doc.page_content not in unique_docs:
            unique_docs[doc.page_content] = doc

    # Return the list of unique documents
    return list(unique_docs.values())


retriever = docsearch.as_retriever(search_kwargs={"k": 30})

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": itemgetter("question")
        | retriever
        | RunnableLambda(deduplicate_docs)
        | RunnableLambda(format_docs),
    }
    | prompt
    | llm
    | StrOutputParser()
)


for chunk in rag_chain.stream(
    {"question": "Quelle est la politique du gouvernement français dans l'IA ?"}
):
    print(chunk, flush=True, end="")

La politique du gouvernement français en matière d'IA s'articule autour de plusieurs axes :

1. Réglementation européenne : La France participe activement à la mise en œuvre de la réglementation européenne sur l'IA, notamment l'[IA Act](https://www.lesechos.fr/thema/articles/ia-leurope-doit-rester-dans-la-course-2105446). 

2. Développement éthique : Le gouvernement soutient le [développement d'une IA éthique et collaborative](https://www.leconomiste.com/article/1122339-intelligence-artificielle-il-n-y-pas-de-strategie-nationale), en ligne avec les valeurs démocratiques.

3. Défense et sécurité : La France investit dans l'IA pour renforcer ses capacités de défense, notamment avec le [développement du supercalculateur le plus rapide d'Europe pour la défense](https://www.msn.com/en-us/news/technology/france-preps-europes-fastest-classified-supercomputer-for-defense-ai/ar-BB1orxT5).

4. Innovation et startups : Le gouvernement soutient l'écosystème des [startups françaises en IA](https://

# With history


In [2]:
%pip install -q langchain langchain-community langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


default_history = get_session_history("default")
default_history.add_user_message("What's a fun fact about Nvidia ? ")
default_history.add_ai_message("""Here's a fun fact about NVIDIA and AI:

NVIDIA's journey from a graphics card manufacturer for PC gaming to an AI powerhouse is quite remarkable. The company's [Graphics Processing Units (GPUs), originally designed for enhancing gaming visuals, have become the backbone of modern AI and deep learning](https://www.newsday.com/business/nvidia-artificial-intelligence-ai-gaming-r29754). This transition has propelled NVIDIA to become [one of the most valuable companies in the world](https://www.msn.com/en-us/money/markets/nvidia-stock-priced-for-perfection-or-poised-for-a-correction/ar-BB1pl0l9), with its chips being integral to almost every major artificial intelligence project.

What's particularly interesting is that NVIDIA's [CUDA (Compute Unified Device Architecture) technology](https://www.msn.com/en-us/money/other/cuda-is-nvidias-secret-sauce-and-now-its-in-the-sights-of-european-regulators/ar-BB1phvnr) has become so crucial to AI development that it's often referred to as the company's "secret sauce." This proprietary parallel computing platform and programming model has given NVIDIA a significant edge in the AI market, to the point where it's now [attracting scrutiny from regulators](https://www.ginjfo.com/actualites/politique-et-economie/nvidia-sous-le-feu-des-projecteurs-de-lautorite-de-la-concurrence-francaise-20240703) due to concerns about the company's market dominance.""")

In [16]:
from datetime import datetime
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain.chains import create_history_aware_retriever

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


from langchain_core.prompts import format_document
from langchain_pinecone import PineconeVectorStore
from langchain_voyageai import VoyageAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_anthropic import ChatAnthropic
from datetime import date
from dotenv import load_dotenv
from langchain_community.chat_message_histories import (
    StreamlitChatMessageHistory,
)


# Load environment variables
load_dotenv()


# Initialize Pinecone and embeddings
EMBEDDING_MODEL = "voyage-large-2-instruct"
INDEX_NAME = "ai-news-index"

embeddings = VoyageAIEmbeddings(model=EMBEDDING_MODEL)  # type: ignore

docsearch = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    text_key="title",
)

# Initialize language model
llm = ChatAnthropic(model_name="claude-3-5-sonnet-20240620", temperature=0.0)  # type: ignore

# Define prompt template
qa_system_prompt = (
    "You are an assistant for question-answering about the latest AI news. "
    "Use the following pieces of retrieved news articles to answer the question. "
    "You are talking to an experienced audience in AI. "
    "If you don't know the answer, just say that you don't know. "
    "Format your answer in markdown and add inline hyperlinks."
    "Do not start your answer with 'Based on the provided context', or similar phrases. "
    f"Today is {date.today()} and below are the latest news on AI. \n"
    "<context>\n"
    "{context}\n"
    "</context>\n\n"
)

qa_prompt = qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ]
)


# Helper functions
def format_docs(docs: list[Document], separator: str = "\n\n") -> str:
    doc_prompt = PromptTemplate.from_template(
        "{page_content} - (Published on {date})\n{url}\n{body}"
    )
    return separator.join(format_document(doc, doc_prompt) for doc in docs)


def convert_docs_dates(docs: list[Document]) -> list[Document]:
    """Convert Unix timestamps to human-readable dates in document metadata."""
    for doc in docs:
        for key in ["date", "found_at"]:
            if key in doc.metadata:
                doc.metadata[key] = datetime.fromtimestamp(doc.metadata[key]).strftime(
                    "%Y-%m-%d"
                )
    return docs


def deduplicate_docs(docs: list[Document]) -> list[Document]:
    unique_docs = {}
    for doc in docs:
        if doc.page_content not in unique_docs:
            unique_docs[doc.page_content] = doc
    return list(unique_docs.values())


# Set up retriever and RAG chain
retriever = docsearch.as_retriever(search_kwargs={"k": 30})

contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


# rag_chain = (
#     {
#         "input": RunnablePassthrough(),
#         "context": history_aware_retriever
#         | convert_docs_dates
#         | deduplicate_docs
#         | format_docs,
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    output_messages_key="answer",
)

batch size None


In [17]:
history_aware_retriever.invoke(
    {
        "chat_history": default_history.messages,
        "history": default_history.messages,
        "input": "What about Apple ?",
    }
)


[Document(page_content="Think You Know Apple? Here's 1 Little-Known Fact You Can't Overlook.", metadata={'body': "This massive enterprise is always in the spotlight, but there's more to the tech titan's story than what meets the eye.", 'date': 1719906420.0, 'found_at': 1719986464.488, 'url': 'https://www.msn.com/en-us/money/other/think-you-know-apple-here-s-1-little-known-fact-you-can-t-overlook/ar-BB1pgwWJ'}),
 Document(page_content="Apple's 10 biggest innovations ever, from the Mac to Apple Intelligence", metadata={'body': 'Apple is having another breakthrough moment in its long history as a technology innovator. The company unveiled its AI project, Apple Intelligence — which will bring ChatGPT and a host of other AI features to the next iPhone,', 'date': 1719903600.0, 'found_at': 1719988051.437, 'url': 'https://www.msn.com/en-us/lifestyle/shopping/apples-10-biggest-innovations-ever-from-the-mac-to-apple-intelligence/ar-BB1pgs5t'}),
 Document(page_content="Apple's 10 biggest innovati

In [ ]:
chain_with_history.invoke(
    {"input": prompt}, config={"configurable": {"session_id": "any"}}
)["answer"]

# TODO :

- [ ] Add rerank https://github.com/pinecone-io/examples/blob/master/learn/generation/better-rag/00-rerankers.ipynb
